# HASTS 416/7 – Stochastic Processes | Group Work Project #1
## Volatility Modeling, Option Pricing & Interest Rate Simulation
---
**Underlying Asset:** SM Energy Company (SM)  
**Current Stock Price:** $232.90 USD  
**Risk-Free Rate:** 1.50% per annum  
**Trading Days per Year:** 250

In [ ]:
# ============================================================
# IMPORTS & GLOBAL SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize, differential_evolution
from scipy.interpolate import CubicSpline
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Global Parameters
S0    = 232.90       # Current stock price
r     = 0.0150       # Annual risk-free rate
T_days_year = 250    # Trading days per year

# Option Data
data = pd.DataFrame({
    'days': [15,15,15,15,15, 60,60,60,60,60, 120,120,120,120,120,
             15,15,15,15,15, 60,60,60,60,60, 120,120,120,120,120],
    'strike': [227.5,230,232.5,235,237.5]*6,
    'price':  [10.52,10.05,7.75,6.01,4.75,
                16.78,17.65,16.86,16.05,15.10,
                27.92,24.12,22.97,21.75,18.06,
                4.32,5.20,6.45,7.56,8.78,
                11.03,12.15,13.37,14.75,15.62,
                14.530629,16.25,17.22,18.74,19.73],
    'type':   ['C']*15 + ['P']*15
})
data['T'] = data['days'] / T_days_year

print('Option Data Loaded:')
print(data.to_string(index=False))

---
## STEP 1 | Sub-group 1 (Members 1–3)
### Task (a): Heston (1993) Model Calibration — Lewis (2001) Approach
**Target Maturity:** 15 days  
**Pricing Method:** Lewis (2001) characteristic function approach  
**Error Function:** Mean Squared Error (MSE)

In [ ]:
# ============================================================
# QUESTION (a): HESTON MODEL — LEWIS (2001) APPROACH
# ============================================================

# --- Heston Characteristic Function ---
def heston_char_func(u, S0, K, T, r, kappa, theta, sigma, rho, v0):
    """
    Heston (1993) characteristic function.
    Parameters: kappa (mean reversion), theta (long-run var), sigma (vol-of-vol),
                rho (correlation), v0 (initial variance).
    """
    i = 1j
    lam = np.sqrt(sigma**2 * (u**2 + i*u) + (kappa - i*rho*sigma*u)**2)
    omega = (kappa - i*rho*sigma*u - lam) / (kappa - i*rho*sigma*u + lam)
    A = i*u*np.log(S0) + i*u*r*T
    B = (kappa*theta/sigma**2) * ((kappa - i*rho*sigma*u - lam)*T - 2*np.log((1 - omega*np.exp(-lam*T))/(1-omega)))
    C = (v0/sigma**2) * (kappa - i*rho*sigma*u - lam) * (1 - np.exp(-lam*T)) / (1 - omega*np.exp(-lam*T))
    return np.exp(A + B + C)

# --- Lewis (2001) Call Price ---
def heston_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0, N=200):
    """
    Lewis (2001) call option price using numerical integration.
    Integrates along the strip Im(u) = 1/2.
    """
    k = np.log(K / (S0 * np.exp(r*T)))  # log-moneyness
    integrand = lambda u: np.real(
        np.exp(-1j * u * k) * heston_char_func(u - 0.5j, S0, K, T, r, kappa, theta, sigma, rho, v0)
        / (u**2 + 0.25)
    )
    # Gauss-Laguerre style: integrate from 0 to inf
    from scipy.integrate import quad
    integral, _ = quad(integrand, 0, 500, limit=500)
    call_price = S0 - (np.sqrt(S0 * K) * np.exp(-r*T) / np.pi) * integral
    return max(call_price, 0.0)

# --- Put via Put-Call Parity ---
def heston_lewis_put(S0, K, T, r, kappa, theta, sigma, rho, v0):
    call = heston_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0)
    put  = call - S0 + K * np.exp(-r * T)
    return max(put, 0.0)

# --- MSE Objective for 15-day maturity ---
def mse_lewis_15(params):
    kappa, theta, sigma, rho, v0 = params
    if kappa <= 0 or theta <= 0 or sigma <= 0 or v0 <= 0 or abs(rho) >= 1:
        return 1e10
    # Feller condition check (optional soft constraint)
    subset = data[data['days'] == 15].copy()
    errors = []
    for _, row in subset.iterrows():
        T = row['T']; K = row['strike']
        try:
            if row['type'] == 'C':
                model_price = heston_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0)
            else:
                model_price = heston_lewis_put(S0, K, T, r, kappa, theta, sigma, rho, v0)
            errors.append((model_price - row['price'])**2)
        except:
            return 1e10
    return np.mean(errors)

# --- Calibration ---
print('Calibrating Heston (Lewis) to 15-day options...')
bounds = [(0.1, 10), (0.01, 1.0), (0.01, 2.0), (-0.99, 0.0), (0.001, 1.0)]
result_lewis = differential_evolution(mse_lewis_15, bounds, seed=42, maxiter=300, tol=1e-8, popsize=15)
kappa_l, theta_l, sigma_l, rho_l, v0_l = result_lewis.x
print(f'\nCalibrated Parameters (Lewis):')
print(f'  kappa (mean reversion speed) = {kappa_l:.4f}')
print(f'  theta (long-run variance)    = {theta_l:.6f}  => long-run vol = {np.sqrt(theta_l)*100:.2f}%')
print(f'  sigma (vol of vol)           = {sigma_l:.4f}')
print(f'  rho   (correlation)          = {rho_l:.4f}')
print(f'  v0    (initial variance)     = {v0_l:.6f}  => initial vol = {np.sqrt(v0_l)*100:.2f}%')
print(f'  MSE                          = {result_lewis.fun:.6f}')
print(f'  Feller condition (2κθ > σ²): {2*kappa_l*theta_l:.4f} > {sigma_l**2:.4f} => {2*kappa_l*theta_l > sigma_l**2}')

In [ ]:
# --- Plot Lewis calibration results ---
subset_15 = data[data['days'] == 15].copy()
model_prices_lewis = []
for _, row in subset_15.iterrows():
    if row['type'] == 'C':
        p = heston_lewis_call(S0, row['strike'], row['T'], r, kappa_l, theta_l, sigma_l, rho_l, v0_l)
    else:
        p = heston_lewis_put(S0, row['strike'], row['T'], r, kappa_l, theta_l, sigma_l, rho_l, v0_l)
    model_prices_lewis.append(p)
subset_15 = subset_15.copy()
subset_15['model_price'] = model_prices_lewis
subset_15['error'] = subset_15['model_price'] - subset_15['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C', 'royalblue', 'o'), ('P', 'tomato', 's')]:
    sub = subset_15[subset_15['type'] == opt_type]
    label = 'Call' if opt_type == 'C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'], marker=marker, linestyle='--', color=color, label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_price'], marker=marker, linestyle='-', color=color, alpha=0.6, label=f'{label} Model')
axes[0].set_title('Heston Lewis (2001): Market vs Model Prices\n(15-day maturity)', fontsize=12)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Option Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(range(len(subset_15)), subset_15['error'],
            color=['royalblue' if t=='C' else 'tomato' for t in subset_15['type']])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Pricing Errors (Model − Market)\nHeston Lewis (2001)', fontsize=12)
axes[1].set_xlabel('Option Index'); axes[1].set_ylabel('Error ($)')
axes[1].set_xticks(range(len(subset_15)))
axes[1].set_xticklabels([f"{r['type']}\nK={r['strike']}" for _, r in subset_15.iterrows()], fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lewis_calibration_15d.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nDetailed Results:')
print(subset_15[['type','strike','price','model_price','error']].to_string(index=False))

---
## STEP 1 | Sub-group 2 (Members 4–6)
### Task (b): Heston (1993) Model Calibration — Carr-Madan (1999) Approach
**Target Maturity:** 15 days  
**Pricing Method:** Carr-Madan FFT approach  
**Error Function:** Mean Squared Error (MSE)

In [ ]:
# ============================================================
# QUESTION (b): HESTON MODEL — CARR-MADAN (1999) APPROACH
# ============================================================

def heston_cf_cm(u, S0, K, T, r, kappa, theta, sigma, rho, v0):
    """
    Heston characteristic function for Carr-Madan (log S_T).
    """
    i = 1j
    xi = kappa - rho * sigma * i * u
    d  = np.sqrt(xi**2 + sigma**2 * (i*u + u**2))
    g  = (xi - d) / (xi + d)
    C  = r*i*u*T + (kappa*theta/sigma**2)*((xi - d)*T - 2*np.log((1 - g*np.exp(-d*T))/(1 - g)))
    D  = ((xi - d)/sigma**2) * ((1 - np.exp(-d*T)) / (1 - g*np.exp(-d*T)))
    return np.exp(C + D*v0 + i*u*np.log(S0))

def carr_madan_call(S0, K, T, r, kappa, theta, sigma, rho, v0, alpha=1.5, N=4096, eta=0.25):
    """
    Carr-Madan (1999) FFT-based call option pricing.
    alpha: damping factor (must be > 0)
    """
    lam = 2 * np.pi / (N * eta)
    b   = np.log(K) - N * lam / 2   # lower bound for log-strike grid
    k_grid = b + lam * np.arange(N)
    
    # Frequency grid
    v = eta * np.arange(N)
    
    # Modified characteristic function
    psi = (np.exp(-r*T) * heston_cf_cm(v - (alpha+1)*1j, S0, K, T, r, kappa, theta, sigma, rho, v0)
           / (alpha**2 + alpha - v**2 + 1j*(2*alpha+1)*v))
    
    # Apply Simpson's rule weights
    w = eta * (3 + (-1)**np.arange(N) - (np.arange(N)==0).astype(float)) / 3
    x = np.exp(1j * b * v) * psi * w
    
    # FFT
    y = np.fft.fft(x)
    calls = (np.exp(-alpha * k_grid) / np.pi) * np.real(y)
    
    # Interpolate at desired strike
    log_K = np.log(K)
    idx = np.searchsorted(k_grid, log_K)
    idx = np.clip(idx, 1, N-2)
    # Linear interpolation
    t  = (log_K - k_grid[idx-1]) / (k_grid[idx] - k_grid[idx-1])
    call_price = calls[idx-1] + t * (calls[idx] - calls[idx-1])
    return max(float(np.real(call_price)), 0.0)

def carr_madan_put(S0, K, T, r, kappa, theta, sigma, rho, v0):
    call = carr_madan_call(S0, K, T, r, kappa, theta, sigma, rho, v0)
    put  = call - S0 + K * np.exp(-r * T)
    return max(put, 0.0)

def mse_cm_15(params):
    kappa, theta, sigma, rho, v0 = params
    if kappa <= 0 or theta <= 0 or sigma <= 0 or v0 <= 0 or abs(rho) >= 1:
        return 1e10
    subset = data[data['days'] == 15].copy()
    errors = []
    for _, row in subset.iterrows():
        T = row['T']; K = row['strike']
        try:
            if row['type'] == 'C':
                mp = carr_madan_call(S0, K, T, r, kappa, theta, sigma, rho, v0)
            else:
                mp = carr_madan_put(S0, K, T, r, kappa, theta, sigma, rho, v0)
            errors.append((mp - row['price'])**2)
        except:
            return 1e10
    return np.mean(errors)

print('Calibrating Heston (Carr-Madan) to 15-day options...')
bounds = [(0.1, 10), (0.01, 1.0), (0.01, 2.0), (-0.99, 0.0), (0.001, 1.0)]
result_cm = differential_evolution(mse_cm_15, bounds, seed=42, maxiter=300, tol=1e-8, popsize=15)
kappa_c, theta_c, sigma_c, rho_c, v0_c = result_cm.x
print(f'\nCalibrated Parameters (Carr-Madan):')
print(f'  kappa = {kappa_c:.4f}')
print(f'  theta = {theta_c:.6f}  => long-run vol = {np.sqrt(theta_c)*100:.2f}%')
print(f'  sigma = {sigma_c:.4f}')
print(f'  rho   = {rho_c:.4f}')
print(f'  v0    = {v0_c:.6f}  => initial vol = {np.sqrt(v0_c)*100:.2f}%')
print(f'  MSE   = {result_cm.fun:.6f}')
print(f'\nComparison Lewis vs Carr-Madan:')
print(f'  kappa: Lewis={kappa_l:.4f}, CM={kappa_c:.4f}, diff={abs(kappa_l-kappa_c):.4f}')
print(f'  theta: Lewis={theta_l:.4f}, CM={theta_c:.4f}, diff={abs(theta_l-theta_c):.4f}')
print(f'  sigma: Lewis={sigma_l:.4f}, CM={sigma_c:.4f}, diff={abs(sigma_l-sigma_c):.4f}')
print(f'  rho:   Lewis={rho_l:.4f}, CM={rho_c:.4f}, diff={abs(rho_l-rho_c):.4f}')
print(f'  v0:    Lewis={v0_l:.4f}, CM={v0_c:.4f}, diff={abs(v0_l-v0_c):.4f}')

In [ ]:
# --- Plot Carr-Madan calibration results ---
subset_15 = data[data['days'] == 15].copy()
model_prices_cm = []
for _, row in subset_15.iterrows():
    if row['type'] == 'C':
        p = carr_madan_call(S0, row['strike'], row['T'], r, kappa_c, theta_c, sigma_c, rho_c, v0_c)
    else:
        p = carr_madan_put(S0, row['strike'], row['T'], r, kappa_c, theta_c, sigma_c, rho_c, v0_c)
    model_prices_cm.append(p)
subset_15 = subset_15.copy()
subset_15['model_price_cm'] = model_prices_cm
subset_15['model_price_lewis'] = model_prices_lewis
subset_15['error_cm'] = subset_15['model_price_cm'] - subset_15['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C', 'royalblue', 'o'), ('P', 'tomato', 's')]:
    sub = subset_15[subset_15['type'] == opt_type]
    label = 'Call' if opt_type == 'C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'],          marker=marker, linestyle='--', color=color,  label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_price_cm'], marker=marker, linestyle='-',  color=color,  alpha=0.6, label=f'{label} CM Model')
    axes[0].plot(sub['strike'], sub['model_price_lewis'], marker=marker, linestyle=':', color='gray', alpha=0.6, label=f'{label} Lewis Model')
axes[0].set_title('Heston Carr-Madan (1999): Market vs Model Prices\n(15-day maturity)', fontsize=12)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Option Price ($)')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

x = np.arange(len(subset_15))
axes[1].bar(x - 0.2, subset_15['error'],    width=0.4, label='Lewis Error',   color='steelblue',  alpha=0.8)
axes[1].bar(x + 0.2, subset_15['error_cm'], width=0.4, label='CM Error',      color='coral',      alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Lewis vs Carr-Madan Pricing Errors\n(15-day maturity)', fontsize=12)
axes[1].set_xlabel('Option Index'); axes[1].set_ylabel('Error ($)')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{r['type']}K={r['strike']}" for _, r in subset_15.iterrows()], fontsize=7, rotation=45)
plt.tight_layout()
plt.savefig('cm_calibration_15d.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 1 | Sub-group 3 (Members 7–10)
### Task (c): Asian Call Option Pricing — Monte Carlo (20-day ATM)
**Instrument:** ATM Asian Call Option, K = S0 = $232.90  
**Maturity:** 20 trading days  
**Method:** Risk-neutral Monte Carlo under Heston dynamics  
**Bank Fee:** 4% markup on fair price

In [ ]:
# ============================================================
# QUESTION (c): ASIAN CALL OPTION PRICING — MONTE CARLO
# ============================================================

# Use Lewis-calibrated parameters (chosen for consistency with Task a)
kappa_use, theta_use, sigma_use, rho_use, v0_use = kappa_l, theta_l, sigma_l, rho_l, v0_l

# Asian option parameters
K_asian = S0          # ATM
T_asian_days = 20
T_asian = T_asian_days / T_days_year
dt = T_asian / T_asian_days
n_sims = 100_000      # 100k Monte Carlo simulations
np.random.seed(42)

def simulate_heston_paths(S0, v0, kappa, theta, sigma, rho, r, T, n_steps, n_sims):
    """
    Euler-Maruyama discretization of Heston dynamics.
    Returns stock price paths: shape (n_sims, n_steps+1)
    """
    dt = T / n_steps
    S = np.zeros((n_sims, n_steps + 1))
    v = np.zeros((n_sims, n_steps + 1))
    S[:, 0] = S0
    v[:, 0] = v0

    for t in range(n_steps):
        Z1 = np.random.standard_normal(n_sims)
        Z2 = np.random.standard_normal(n_sims)
        Zv = Z1
        Zs = rho * Z1 + np.sqrt(1 - rho**2) * Z2
        v_pos = np.maximum(v[:, t], 0)  # Reflect to avoid negative variance
        v[:, t+1] = v_pos + kappa * (theta - v_pos) * dt + sigma * np.sqrt(v_pos * dt) * Zv
        v[:, t+1] = np.maximum(v[:, t+1], 0)
        S[:, t+1] = S[:, t] * np.exp((r - 0.5 * v_pos) * dt + np.sqrt(v_pos * dt) * Zs)
    return S, v

print(f'Running {n_sims:,} Monte Carlo simulations for Asian Call...')
print(f'Parameters: K=${K_asian}, T={T_asian_days} days, S0=${S0}')

S_paths, v_paths = simulate_heston_paths(
    S0, v0_use, kappa_use, theta_use, sigma_use, rho_use,
    r, T_asian, T_asian_days, n_sims
)

# Asian payoff: average includes S0 at t=0
avg_prices = np.mean(S_paths, axis=1)  # shape (n_sims,)
payoffs = np.maximum(avg_prices - K_asian, 0)
discount_factor = np.exp(-r * T_asian)

fair_price = discount_factor * np.mean(payoffs)
std_err = discount_factor * np.std(payoffs) / np.sqrt(n_sims)
ci_lower = fair_price - 1.96 * std_err
ci_upper = fair_price + 1.96 * std_err

bank_fee = 0.04
client_price = fair_price * (1 + bank_fee)

print(f'\n=== ASIAN CALL OPTION RESULTS ===')
print(f'  Fair Price (Monte Carlo):    ${fair_price:.4f}')
print(f'  Standard Error:              ${std_err:.4f}')
print(f'  95% Confidence Interval:     [${ci_lower:.4f}, ${ci_upper:.4f}]')
print(f'  Bank Fee (4%):               ${fair_price * bank_fee:.4f}')
print(f'  Final Client Price:          ${client_price:.4f}')

In [ ]:
# --- Visualize Asian option simulation ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Sample paths
n_plot = 200
t_grid = np.linspace(0, T_asian_days, T_asian_days + 1)
for i in range(n_plot):
    axes[0].plot(t_grid, S_paths[i], alpha=0.1, color='steelblue', linewidth=0.5)
axes[0].axhline(K_asian, color='red', linestyle='--', linewidth=1.5, label=f'Strike K=${K_asian:.0f}')
axes[0].set_title(f'Sample Heston Stock Paths (first {n_plot})', fontsize=11)
axes[0].set_xlabel('Trading Days'); axes[0].set_ylabel('Stock Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# 2. Payoff distribution
axes[1].hist(payoffs[payoffs > 0], bins=80, color='seagreen', edgecolor='white', alpha=0.8)
axes[1].axvline(np.mean(payoffs[payoffs > 0]), color='red', linestyle='--', label=f'Mean payoff (ITM): ${np.mean(payoffs[payoffs>0]):.2f}')
axes[1].set_title('Distribution of In-the-Money Payoffs\n(Asian Call)', fontsize=11)
axes[1].set_xlabel('Payoff ($)'); axes[1].set_ylabel('Frequency')
axes[1].legend(); axes[1].grid(alpha=0.3)

# 3. Convergence of MC estimate
batch_sizes = np.logspace(2, np.log10(n_sims), 100).astype(int)
conv_prices = [discount_factor * np.mean(np.maximum(np.mean(S_paths[:b], axis=1) - K_asian, 0)) for b in batch_sizes]
axes[2].semilogx(batch_sizes, conv_prices, color='darkorange')
axes[2].axhline(fair_price, color='red', linestyle='--', label=f'Final estimate: ${fair_price:.4f}')
axes[2].fill_between(batch_sizes, ci_lower, ci_upper, alpha=0.2, color='red', label='95% CI')
axes[2].set_title('Monte Carlo Convergence\n(Asian Call Price)', fontsize=11)
axes[2].set_xlabel('Number of Simulations'); axes[2].set_ylabel('Estimated Price ($)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'ATM Asian Call Option | 20 Days | Heston Dynamics', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('asian_call_mc.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSummary for client:')
print(f'  Fair value (theoretical): ${fair_price:.2f}')
print(f'  Price charged to client:  ${client_price:.2f} (inclusive of 4% service fee)')
pct_itm = 100 * np.mean(payoffs > 0)
print(f'  Probability of being ITM at expiry: {pct_itm:.1f}%')

---
## STEP 2 | Sub-group 3 (Members 7–10)
### Task (d): Bates (1996) Model Calibration — Lewis Approach (60-day maturity)

In [ ]:
# ============================================================
# QUESTION (d): BATES (1996) MODEL — LEWIS APPROACH — 60 DAYS
# ============================================================

def bates_char_func(u, S0, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    """
    Bates (1996) characteristic function = Heston + compound Poisson jumps.
    lam_j: jump intensity (jumps per year)
    mu_j:  mean jump size (log)
    delta_j: std of jump size (log)
    """
    i = 1j
    # Heston part
    xi  = kappa - rho * sigma * i * u
    d   = np.sqrt(xi**2 + sigma**2 * (u**2 + i*u))
    g   = (xi - d) / (xi + d)
    Cv  = (kappa*theta/sigma**2) * ((xi - d)*T - 2*np.log((1 - g*np.exp(-d*T))/(1 - g)))
    Dv  = ((xi - d)/sigma**2) * (1 - np.exp(-d*T)) / (1 - g*np.exp(-d*T))
    # Jump part (Merton lognormal jumps)
    k_bar = np.exp(mu_j + 0.5*delta_j**2) - 1  # mean relative jump
    Cj  = lam_j * T * (np.exp(i*u*mu_j - 0.5*delta_j**2*u**2) - 1 - i*u*k_bar)
    return np.exp(i*u*(np.log(S0) + (r - lam_j*k_bar)*T) + Cv + Dv*v0 + Cj)

def bates_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    from scipy.integrate import quad
    k = np.log(K / (S0 * np.exp(r*T)))
    k_bar = np.exp(mu_j + 0.5*delta_j**2) - 1
    def integrand(u):
        cf = bates_char_func(u - 0.5j, S0, T, r - lam_j*k_bar, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
        return np.real(np.exp(-1j*u*k) * cf / (u**2 + 0.25))
    integral, _ = quad(integrand, 0, 500, limit=500)
    call = S0 - (np.sqrt(S0*K) * np.exp(-r*T) / np.pi) * integral
    return max(float(call), 0.0)

def bates_lewis_put(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    call = bates_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
    return max(call - S0 + K*np.exp(-r*T), 0.0)

def mse_bates_lewis_60(params):
    kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j = params
    if kappa<=0 or theta<=0 or sigma<=0 or v0<=0 or abs(rho)>=1 or lam_j<0 or delta_j<=0:
        return 1e10
    subset = data[data['days'] == 60].copy()
    errors = []
    for _, row in subset.iterrows():
        T = row['T']; K = row['strike']
        try:
            if row['type'] == 'C':
                mp = bates_lewis_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
            else:
                mp = bates_lewis_put(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
            errors.append((mp - row['price'])**2)
        except:
            return 1e10
    return np.mean(errors)

print('Calibrating Bates (Lewis) to 60-day options...')
bounds_bates = [(0.1,10),(0.001,1),(0.01,2),(-0.99,0),(0.001,1),(0,5),(-0.5,0.5),(0.001,0.5)]
result_bates_l = differential_evolution(mse_bates_lewis_60, bounds_bates, seed=42, maxiter=400, tol=1e-8, popsize=20)
kappa_bl, theta_bl, sigma_bl, rho_bl, v0_bl, lam_bl, mu_bl, delta_bl = result_bates_l.x
print(f'\nBates (Lewis) Calibrated Parameters — 60 days:')
print(f'  kappa  = {kappa_bl:.4f}')
print(f'  theta  = {theta_bl:.6f}  (long-run vol = {np.sqrt(theta_bl)*100:.2f}%)')
print(f'  sigma  = {sigma_bl:.4f}')
print(f'  rho    = {rho_bl:.4f}')
print(f'  v0     = {v0_bl:.6f}  (initial vol = {np.sqrt(v0_bl)*100:.2f}%)')
print(f'  lambda = {lam_bl:.4f}  (jumps/year)')
print(f'  mu_j   = {mu_bl:.4f}  (mean log jump)')
print(f'  delta  = {delta_bl:.4f}  (jump vol)')
print(f'  MSE    = {result_bates_l.fun:.6f}')

In [ ]:
# --- Plot Bates (Lewis) calibration ---
subset_60 = data[data['days'] == 60].copy()
bates_prices_l = []
for _, row in subset_60.iterrows():
    if row['type'] == 'C':
        p = bates_lewis_call(S0, row['strike'], row['T'], r, kappa_bl, theta_bl, sigma_bl, rho_bl, v0_bl, lam_bl, mu_bl, delta_bl)
    else:
        p = bates_lewis_put(S0, row['strike'], row['T'], r, kappa_bl, theta_bl, sigma_bl, rho_bl, v0_bl, lam_bl, mu_bl, delta_bl)
    bates_prices_l.append(p)
subset_60['model_bates_l'] = bates_prices_l
subset_60['error_bates_l'] = subset_60['model_bates_l'] - subset_60['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C','royalblue','o'),('P','tomato','s')]:
    sub = subset_60[subset_60['type']==opt_type]
    label = 'Call' if opt_type=='C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'],         marker=marker, linestyle='--', color=color, label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_bates_l'], marker=marker, linestyle='-',  color=color, alpha=0.6, label=f'{label} Bates-Lewis')
axes[0].set_title('Bates (1996) Lewis Approach: Market vs Model\n(60-day maturity)', fontsize=12)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Option Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].bar(range(len(subset_60)), subset_60['error_bates_l'],
            color=['royalblue' if t=='C' else 'tomato' for t in subset_60['type']])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Pricing Errors — Bates Lewis (60 days)', fontsize=12)
axes[1].set_xlabel('Option'); axes[1].set_ylabel('Error ($)')
axes[1].set_xticks(range(len(subset_60)))
axes[1].set_xticklabels([f"{r['type']}K={r['strike']}" for _,r in subset_60.iterrows()], fontsize=8, rotation=45)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('bates_lewis_60d.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 2 | Sub-group 1 (Members 1–3)
### Task (e): Bates (1996) Model Calibration — Carr-Madan Approach (60-day maturity)

In [ ]:
# ============================================================
# QUESTION (e): BATES (1996) — CARR-MADAN APPROACH — 60 DAYS
# ============================================================

def bates_cf_cm(u, S0, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    """
    Bates CF for Carr-Madan (in log S_T domain).
    """
    i = 1j
    xi  = kappa - rho * sigma * i * u
    d   = np.sqrt(xi**2 + sigma**2*(i*u + u**2))
    g   = (xi - d)/(xi + d)
    C   = r*i*u*T + (kappa*theta/sigma**2)*((xi-d)*T - 2*np.log((1 - g*np.exp(-d*T))/(1-g)))
    D   = ((xi-d)/sigma**2)*(1-np.exp(-d*T))/(1-g*np.exp(-d*T))
    k_bar = np.exp(mu_j + 0.5*delta_j**2) - 1
    Cj  = lam_j*T*(np.exp(i*u*mu_j - 0.5*delta_j**2*u**2) - 1 - i*u*k_bar)
    return np.exp(C + D*v0 + i*u*np.log(S0) - i*u*lam_j*k_bar*T + Cj)

def bates_cm_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j, alpha=1.5, N=4096, eta=0.25):
    lam = 2 * np.pi / (N * eta)
    b   = np.log(K) - N * lam / 2
    k_grid = b + lam * np.arange(N)
    v = eta * np.arange(N)
    psi = (np.exp(-r*T) * bates_cf_cm(v - (alpha+1)*1j, S0, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
           / (alpha**2 + alpha - v**2 + 1j*(2*alpha+1)*v))
    w   = eta * (3 + (-1)**np.arange(N) - (np.arange(N)==0).astype(float)) / 3
    x   = np.exp(1j * b * v) * psi * w
    y   = np.fft.fft(x)
    calls = (np.exp(-alpha * k_grid) / np.pi) * np.real(y)
    log_K = np.log(K)
    idx = np.searchsorted(k_grid, log_K)
    idx = np.clip(idx, 1, N-2)
    t   = (log_K - k_grid[idx-1]) / (k_grid[idx] - k_grid[idx-1])
    return max(float(np.real(calls[idx-1] + t*(calls[idx]-calls[idx-1]))), 0.0)

def bates_cm_put(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j):
    call = bates_cm_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
    return max(call - S0 + K*np.exp(-r*T), 0.0)

def mse_bates_cm_60(params):
    kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j = params
    if kappa<=0 or theta<=0 or sigma<=0 or v0<=0 or abs(rho)>=1 or lam_j<0 or delta_j<=0:
        return 1e10
    subset = data[data['days'] == 60].copy()
    errors = []
    for _, row in subset.iterrows():
        T = row['T']; K = row['strike']
        try:
            if row['type'] == 'C':
                mp = bates_cm_call(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
            else:
                mp = bates_cm_put(S0, K, T, r, kappa, theta, sigma, rho, v0, lam_j, mu_j, delta_j)
            errors.append((mp - row['price'])**2)
        except:
            return 1e10
    return np.mean(errors)

print('Calibrating Bates (Carr-Madan) to 60-day options...')
result_bates_cm = differential_evolution(mse_bates_cm_60, bounds_bates, seed=42, maxiter=400, tol=1e-8, popsize=20)
kappa_bc, theta_bc, sigma_bc, rho_bc, v0_bc, lam_bc, mu_bc, delta_bc = result_bates_cm.x
print(f'\nBates (Carr-Madan) Calibrated Parameters — 60 days:')
print(f'  kappa  = {kappa_bc:.4f}')
print(f'  theta  = {theta_bc:.6f}  (long-run vol = {np.sqrt(theta_bc)*100:.2f}%)')
print(f'  sigma  = {sigma_bc:.4f}')
print(f'  rho    = {rho_bc:.4f}')
print(f'  v0     = {v0_bc:.6f}  (initial vol = {np.sqrt(v0_bc)*100:.2f}%)')
print(f'  lambda = {lam_bc:.4f}  (jumps/year)')
print(f'  mu_j   = {mu_bc:.4f}')
print(f'  delta  = {delta_bc:.4f}')
print(f'  MSE    = {result_bates_cm.fun:.6f}')

print(f'\nComparison Bates Lewis vs Carr-Madan (60d):')
names = ['kappa','theta','sigma','rho','v0','lambda','mu_j','delta']
for nm, bl, bc in zip(names, result_bates_l.x, result_bates_cm.x):
    print(f'  {nm:6s}: Lewis={bl:.4f}, CM={bc:.4f}')

In [ ]:
# --- Plot comparison Bates Lewis vs Carr-Madan ---
bates_prices_cm = []
for _, row in subset_60.iterrows():
    if row['type'] == 'C':
        p = bates_cm_call(S0, row['strike'], row['T'], r, kappa_bc, theta_bc, sigma_bc, rho_bc, v0_bc, lam_bc, mu_bc, delta_bc)
    else:
        p = bates_cm_put(S0, row['strike'], row['T'], r, kappa_bc, theta_bc, sigma_bc, rho_bc, v0_bc, lam_bc, mu_bc, delta_bc)
    bates_prices_cm.append(p)
subset_60['model_bates_cm'] = bates_prices_cm
subset_60['error_bates_cm'] = subset_60['model_bates_cm'] - subset_60['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt_type, color, marker in [('C','royalblue','o'),('P','tomato','s')]:
    sub = subset_60[subset_60['type']==opt_type]
    label = 'Call' if opt_type=='C' else 'Put'
    axes[0].plot(sub['strike'], sub['price'],           marker=marker, linestyle='--', color=color, label=f'{label} Market')
    axes[0].plot(sub['strike'], sub['model_bates_l'],   marker=marker, linestyle='-',  color=color, alpha=0.7, label=f'{label} Lewis')
    axes[0].plot(sub['strike'], sub['model_bates_cm'],  marker=marker, linestyle=':',  color=color, alpha=0.7, label=f'{label} CM')
axes[0].set_title('Bates 60d: Market vs Lewis vs Carr-Madan', fontsize=11)
axes[0].set_xlabel('Strike ($)'); axes[0].set_ylabel('Price ($)')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

x = np.arange(len(subset_60))
axes[1].bar(x-0.2, subset_60['error_bates_l'],  width=0.4, label='Lewis Error', color='steelblue', alpha=0.8)
axes[1].bar(x+0.2, subset_60['error_bates_cm'], width=0.4, label='CM Error',    color='coral',     alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Bates 60d: Pricing Errors Comparison', fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{r['type']}K={r['strike']}" for _,r in subset_60.iterrows()], fontsize=7, rotation=45)
plt.tight_layout()
plt.savefig('bates_cm_60d.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 2 | Sub-group 2 (Members 4–6)
### Task (f): Put Option Pricing — Monte Carlo (70 days, 95% Moneyness)

In [ ]:
# ============================================================
# QUESTION (f): PUT OPTION — MONTE CARLO — 70 DAYS, 95% MONEYNESS
# ============================================================

# Use Bates (Lewis) calibrated params from 60-day calibration
K_put   = 0.95 * S0          # 95% moneyness
T_put_days = 70
T_put   = T_put_days / T_days_year
n_sims_put = 100_000
np.random.seed(123)

print(f'Put Option Parameters:')
print(f'  S0 = ${S0}, K = ${K_put:.4f} (95% moneyness), T = {T_put_days} days')

def simulate_bates_paths(S0, v0, kappa, theta, sigma, rho, r, lam_j, mu_j, delta_j, T, n_steps, n_sims):
    """
    Euler-Maruyama for Bates model (Heston + Merton jumps).
    """
    dt = T / n_steps
    S  = np.zeros((n_sims, n_steps+1))
    v  = np.zeros((n_sims, n_steps+1))
    S[:, 0] = S0
    v[:, 0] = v0
    k_bar = np.exp(mu_j + 0.5*delta_j**2) - 1

    for t in range(n_steps):
        Z1 = np.random.standard_normal(n_sims)
        Z2 = np.random.standard_normal(n_sims)
        Zv = Z1
        Zs = rho * Z1 + np.sqrt(1 - rho**2) * Z2
        v_pos = np.maximum(v[:, t], 0)
        # Jump component
        n_jumps = np.random.poisson(lam_j * dt, n_sims)
        jump_sum = np.array([np.sum(np.random.normal(mu_j, delta_j, nj)) if nj > 0 else 0.0 for nj in n_jumps])
        jump_factor = np.exp(jump_sum)
        v[:, t+1] = np.maximum(v_pos + kappa*(theta - v_pos)*dt + sigma*np.sqrt(v_pos*dt)*Zv, 0)
        S[:, t+1] = S[:, t] * np.exp((r - lam_j*k_bar - 0.5*v_pos)*dt + np.sqrt(v_pos*dt)*Zs) * jump_factor
    return S, v

print(f'\nRunning {n_sims_put:,} simulations for Put option...')
S_put, _ = simulate_bates_paths(
    S0, v0_bl, kappa_bl, theta_bl, sigma_bl, rho_bl, r,
    lam_bl, mu_bl, delta_bl, T_put, T_put_days, n_sims_put
)

# European Put payoff on final price
put_payoffs    = np.maximum(K_put - S_put[:, -1], 0)
put_fair_price = np.exp(-r * T_put) * np.mean(put_payoffs)
put_std_err    = np.exp(-r * T_put) * np.std(put_payoffs) / np.sqrt(n_sims_put)
put_ci         = (put_fair_price - 1.96*put_std_err, put_fair_price + 1.96*put_std_err)

print(f'\n=== PUT OPTION RESULTS ===')
print(f'  Strike (95% moneyness):      ${K_put:.4f}')
print(f'  Fair Price (Monte Carlo):    ${put_fair_price:.4f}')
print(f'  Standard Error:              ${put_std_err:.4f}')
print(f'  95% CI:                      [${put_ci[0]:.4f}, ${put_ci[1]:.4f}]')
pct_itm_put = 100 * np.mean(put_payoffs > 0)
print(f'  Probability ITM:             {pct_itm_put:.1f}%')

In [ ]:
# --- Put option plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

n_plot = 200
t_grid = np.linspace(0, T_put_days, T_put_days + 1)
for i in range(n_plot):
    axes[0].plot(t_grid, S_put[i], alpha=0.1, color='steelblue', linewidth=0.5)
axes[0].axhline(K_put, color='red', linestyle='--', linewidth=1.5, label=f'Strike K=${K_put:.0f}')
axes[0].axhline(S0,    color='green', linestyle=':', linewidth=1.5, label=f'S0=${S0}')
axes[0].set_title(f'Bates Stock Paths (first {n_plot})\nPut Option Pricing', fontsize=11)
axes[0].set_xlabel('Trading Days'); axes[0].set_ylabel('Price ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

final_prices = S_put[:, -1]
axes[1].hist(final_prices, bins=100, color='steelblue', edgecolor='white', alpha=0.8, density=True)
axes[1].axvline(K_put, color='red', linestyle='--', label=f'Strike K=${K_put:.0f}')
axes[1].axvline(S0,    color='green', linestyle=':', label=f'S0=${S0}')
axes[1].set_title('Distribution of Final Stock Prices\n(70-day Bates)', fontsize=11)
axes[1].set_xlabel('S_T ($)'); axes[1].set_ylabel('Density')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Convergence
batch = np.logspace(2, np.log10(n_sims_put), 100).astype(int)
conv  = [np.exp(-r*T_put) * np.mean(np.maximum(K_put - S_put[:b,-1], 0)) for b in batch]
axes[2].semilogx(batch, conv, color='darkorange')
axes[2].axhline(put_fair_price, color='red', linestyle='--', label=f'Estimate: ${put_fair_price:.4f}')
axes[2].set_title('MC Convergence — Put Option', fontsize=11)
axes[2].set_xlabel('Simulations'); axes[2].set_ylabel('Estimated Price ($)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'SM Put Option | K=${K_put:.2f} (95%) | 70 Days | Bates Model', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('put_option_mc.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 3 | Full Team
### Task (g): CIR (1985) Model Calibration — Euribor Term Structure

In [ ]:
# ============================================================
# QUESTION (g): CIR MODEL CALIBRATION — EURIBOR
# ============================================================

# Market Euribor data
euribor_tenors_days = np.array([7, 30, 90, 180, 360])  # in days
euribor_tenors_yr   = euribor_tenors_days / 365.0
euribor_rates       = np.array([0.648, 0.679, 1.173, 1.809, 2.556]) / 100  # as decimals

print('Market Euribor Term Structure:')
for tenor, rate in zip(['1W','1M','3M','6M','12M'], euribor_rates):
    print(f'  Euribor {tenor}: {rate*100:.3f}%')

# --- Cubic Spline Interpolation ---
cs = CubicSpline(euribor_tenors_yr, euribor_rates, bc_type='not-a-knot')
weeks_in_year = 52
t_weekly = np.arange(1, weeks_in_year + 1) / weeks_in_year  # weekly for 1 year
r_weekly = cs(t_weekly)
r_weekly = np.maximum(r_weekly, 0.001)  # floor at 0.1%

print(f'\nCubic Spline — First 5 weekly rates:')
for i in range(5):
    print(f'  Week {i+1} ({t_weekly[i]*52:.0f}w): {r_weekly[i]*100:.4f}%')

# --- CIR Bond Price (Zero-Coupon) ---
def cir_bond_price(T, r0, kappa, theta, sigma):
    """
    CIR (1985) zero-coupon bond price P(0,T).
    """
    gamma = np.sqrt(kappa**2 + 2*sigma**2)
    A_num = 2*gamma*np.exp((kappa+gamma)*T/2)
    A_den = (kappa+gamma)*(np.exp(gamma*T) - 1) + 2*gamma
    A     = (A_num / A_den) ** (2*kappa*theta/sigma**2)
    B_num = 2*(np.exp(gamma*T) - 1)
    B     = B_num / ((kappa+gamma)*(np.exp(gamma*T)-1) + 2*gamma)
    return A * np.exp(-B * r0)

def cir_yield(T, r0, kappa, theta, sigma):
    P = cir_bond_price(T, r0, kappa, theta, sigma)
    return -np.log(P) / T

# --- MSE Calibration to splined weekly rates ---
r0_cir = euribor_rates[0]  # current short rate ~ 1-week Euribor

def mse_cir(params):
    kappa, theta, sigma = params
    if kappa <= 0 or theta <= 0 or sigma <= 0:
        return 1e10
    if 2*kappa*theta <= sigma**2:  # Feller condition
        return 1e10
    errors = []
    for t, r_mkt in zip(t_weekly, r_weekly):
        r_model = cir_yield(t, r0_cir, kappa, theta, sigma)
        errors.append((r_model - r_mkt)**2)
    return np.mean(errors)

print('\nCalibrating CIR model to Euribor term structure...')
bounds_cir = [(0.01, 5.0), (0.001, 0.2), (0.001, 0.5)]
result_cir = differential_evolution(mse_cir, bounds_cir, seed=42, maxiter=500, tol=1e-12, popsize=20)
kappa_cir, theta_cir, sigma_cir = result_cir.x

print(f'\nCIR Calibrated Parameters:')
print(f'  kappa (speed of mean reversion) = {kappa_cir:.4f}')
print(f'  theta (long-run rate)           = {theta_cir*100:.4f}%')
print(f'  sigma (vol of short rate)       = {sigma_cir:.4f}')
print(f'  r0    (initial rate)            = {r0_cir*100:.4f}%')
print(f'  MSE                             = {result_cir.fun:.10f}')
print(f'  Feller condition (2κθ > σ²):    {2*kappa_cir*theta_cir:.4f} > {sigma_cir**2:.4f} => {2*kappa_cir*theta_cir > sigma_cir**2}')

In [ ]:
# --- CIR fit plot ---
t_fine    = np.linspace(1/365, 1.0, 200)
r_cir_fit = np.array([cir_yield(t, r0_cir, kappa_cir, theta_cir, sigma_cir) for t in t_fine])
r_weekly_fit = np.array([cir_yield(t, r0_cir, kappa_cir, theta_cir, sigma_cir) for t in t_weekly])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t_fine*12, r_cir_fit*100,  color='royalblue', linewidth=2, label='CIR Model Fit')
axes[0].plot(t_weekly*12, r_weekly*100, color='gray', linestyle='--', linewidth=1.2, label='Cubic Spline (Market)')
axes[0].scatter(euribor_tenors_yr*12, euribor_rates*100, color='red', zorder=5, s=80, label='Market Euribor')
axes[0].set_title('CIR Model vs Market Term Structure\n(Euribor)', fontsize=12)
axes[0].set_xlabel('Maturity (months)'); axes[0].set_ylabel('Rate (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

residuals = (r_weekly_fit - r_weekly) * 100
axes[1].bar(range(len(t_weekly)), residuals, color='steelblue', alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('CIR Fitting Residuals (Model − Market)\n(basis points, %)', fontsize=12)
axes[1].set_xlabel('Week'); axes[1].set_ylabel('Residual (% rate)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cir_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'CIR fit at market tenors:')
for tenor, T, r_mkt in zip(['1W','1M','3M','6M','12M'], euribor_tenors_yr, euribor_rates):
    r_mod = cir_yield(T, r0_cir, kappa_cir, theta_cir, sigma_cir)
    print(f'  {tenor}: Market={r_mkt*100:.4f}%, CIR={r_mod*100:.4f}%, Error={abs(r_mod-r_mkt)*10000:.2f}bps')

---
## STEP 3 | Full Team
### Task (h): CIR — Monte Carlo Simulation of 12-Month Euribor (100,000 paths)

In [ ]:
# ============================================================
# QUESTION (h): CIR MONTE CARLO — 12-MONTH EURIBOR SIMULATION
# ============================================================

n_sims_cir  = 100_000
n_days_cir  = 365
dt_cir      = 1 / 365
np.random.seed(42)

print(f'Running {n_sims_cir:,} CIR simulations ({n_days_cir} daily steps)...')

# Store only the 12M Euribor at each time step (memory efficient)
r_cir = np.full((n_sims_cir, n_days_cir + 1), r0_cir)

for t in range(n_days_cir):
    Z = np.random.standard_normal(n_sims_cir)
    r_pos = np.maximum(r_cir[:, t], 0)
    dr    = kappa_cir * (theta_cir - r_pos) * dt_cir + sigma_cir * np.sqrt(r_pos * dt_cir) * Z
    r_cir[:, t+1] = np.maximum(r_cir[:, t] + dr, 0)

# Terminal distribution (r at 1 year)
r_terminal = r_cir[:, -1]

# Use 12-month Euribor (approximate as terminal short rate)
confidence = 0.95
alpha_tail  = (1 - confidence) / 2
r_low  = np.quantile(r_terminal, alpha_tail)
r_high = np.quantile(r_terminal, 1 - alpha_tail)
r_exp  = np.mean(r_terminal)

current_12m_euribor = euribor_rates[-1]  # 2.556%

print(f'\n=== CIR SIMULATION RESULTS (12M Euribor in 1 Year) ===')
print(f'  Confidence Level:            {confidence*100:.0f}%')
print(f'  Min ({alpha_tail*100:.1f}% quantile):         {r_low*100:.4f}%')
print(f'  Max ({(1-alpha_tail)*100:.1f}% quantile):        {r_high*100:.4f}%')
print(f'  Expected Value (1 year):     {r_exp*100:.4f}%')
print(f'  Current 12M Euribor:         {current_12m_euribor*100:.4f}%')
print(f'  Expected Change:             {(r_exp - current_12m_euribor)*100:.4f}% ({(r_exp-current_12m_euribor)/current_12m_euribor*100:.1f}% relative change)')
print(f'\n  Pricing Impact Discussion:')
if r_exp > current_12m_euribor:
    print(f'  Higher rates in 1 year vs today => call option prices would be slightly higher')
    print(f'  (higher discount rate increases carry cost), put option prices decrease.')
else:
    print(f'  Lower rates in 1 year => call option prices slightly lower, put prices increase.')

In [ ]:
# --- CIR Monte Carlo Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Sample paths
t_plot = np.linspace(0, 365, n_days_cir + 1)
n_show = 500
for i in range(n_show):
    axes[0].plot(t_plot, r_cir[i]*100, alpha=0.03, color='steelblue', linewidth=0.5)
# Mean path
axes[0].plot(t_plot, np.mean(r_cir, axis=0)*100, color='red', linewidth=2, label='Mean path')
axes[0].plot(t_plot, np.quantile(r_cir, 0.025, axis=0)*100, color='orange', linewidth=1.5, linestyle='--', label='95% CI bounds')
axes[0].plot(t_plot, np.quantile(r_cir, 0.975, axis=0)*100, color='orange', linewidth=1.5, linestyle='--')
axes[0].set_title(f'CIR Rate Paths ({n_sims_cir:,} simulations)', fontsize=11)
axes[0].set_xlabel('Days'); axes[0].set_ylabel('Rate (%)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# 2. Terminal distribution
axes[1].hist(r_terminal*100, bins=150, color='steelblue', edgecolor='white', alpha=0.8, density=True)
axes[1].axvline(r_exp*100,  color='red',    linestyle='-',  linewidth=2, label=f'E[r] = {r_exp*100:.3f}%')
axes[1].axvline(r_low*100,  color='orange', linestyle='--', linewidth=1.5, label=f'95% CI: [{r_low*100:.3f}%, {r_high*100:.3f}%]')
axes[1].axvline(r_high*100, color='orange', linestyle='--', linewidth=1.5)
axes[1].axvline(current_12m_euribor*100, color='green', linestyle=':', linewidth=2, label=f'Current 12M: {current_12m_euribor*100:.3f}%')
axes[1].set_title('Distribution of 12M Euribor in 1 Year\n(CIR Monte Carlo)', fontsize=11)
axes[1].set_xlabel('Rate (%)'); axes[1].set_ylabel('Density')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# 3. Time evolution of percentiles
pcts = [5, 25, 50, 75, 95]
colors_p = ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd']
for pct, c in zip(pcts, colors_p):
    axes[2].plot(t_plot, np.quantile(r_cir, pct/100, axis=0)*100, label=f'P{pct}', color=c, linewidth=1.5)
axes[2].set_title('CIR Percentile Bands (1 Year)', fontsize=11)
axes[2].set_xlabel('Days'); axes[2].set_ylabel('Rate (%)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('CIR (1985) — Euribor 12-Month Rate Simulation', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('cir_mc_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFinal Summary Table:')
print(f'{"Metric":<35} {"Value":>15}')
print('-'*52)
print(f'{"Expected 12M Euribor (1 year)":<35} {r_exp*100:>14.4f}%')
print(f'{"95% CI Lower":<35} {r_low*100:>14.4f}%')
print(f'{"95% CI Upper":<35} {r_high*100:>14.4f}%')
print(f'{"Current 12M Euribor":<35} {current_12m_euribor*100:>14.4f}%')
print(f'{"Probability rate falls":<35} {np.mean(r_terminal < current_12m_euribor)*100:>14.1f}%')
print(f'{"Probability rate rises":<35} {np.mean(r_terminal > current_12m_euribor)*100:>14.1f}%')

---
## SUMMARY OF ALL RESULTS

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================
print('='*65)
print('HASTS 416/7 — GWP1 COMPLETE RESULTS SUMMARY')
print('='*65)
print(f'\n[Step 1a] Heston Lewis (15d): κ={kappa_l:.4f}, θ={theta_l:.4f}, σ={sigma_l:.4f}, ρ={rho_l:.4f}, v0={v0_l:.4f}')
print(f'[Step 1b] Heston CM (15d):    κ={kappa_c:.4f}, θ={theta_c:.4f}, σ={sigma_c:.4f}, ρ={rho_c:.4f}, v0={v0_c:.4f}')
print(f'[Step 1c] ATM Asian Call (20d): Fair=${fair_price:.4f}, Client=${client_price:.4f}')
print(f'[Step 2d] Bates Lewis (60d):  κ={kappa_bl:.4f}, θ={theta_bl:.4f}, σ={sigma_bl:.4f}, ρ={rho_bl:.4f}')
print(f'          Jump params: λ={lam_bl:.4f}, μ={mu_bl:.4f}, δ={delta_bl:.4f}')
print(f'[Step 2e] Bates CM (60d):     κ={kappa_bc:.4f}, θ={theta_bc:.4f}, σ={sigma_bc:.4f}, ρ={rho_bc:.4f}')
print(f'          Jump params: λ={lam_bc:.4f}, μ={mu_bc:.4f}, δ={delta_bc:.4f}')
print(f'[Step 2f] Put Option (70d, 95%): Fair=${put_fair_price:.4f}, 95%CI=[${put_ci[0]:.4f},${put_ci[1]:.4f}]')
print(f'[Step 3g] CIR Calibration:    κ={kappa_cir:.4f}, θ={theta_cir*100:.4f}%, σ={sigma_cir:.4f}')
print(f'[Step 3h] 12M Euribor in 1yr: E[r]={r_exp*100:.4f}%, 95%CI=[{r_low*100:.4f}%,{r_high*100:.4f}%]')
print('='*65)